# HPO Baseline (v2) - parameters derived, not searched

The baseline is a plain XLM-R-large sequence classifier: it has no fusion
head to tune. To keep the training setup identical to the dual-view variants
(so that any difference reflects the architecture, not the tuning), it takes
the fixed encoder hyperparameters and the epoch count chosen on
`dual_view_v2`. No search runs here.

**Output:** `hpo/baseline_v2/best_params.json`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
import utils_split as u

In [ ]:
# The average-fusion variant is the only one that is searched. Its result is
# read here and reused, so all three architectures share one training setup.
DV_PATH = best_params_path("dual_view_v2")
assert os.path.exists(DV_PATH), (
    "hpo/dual_view_v2/best_params.json is missing - run hpo_kusa_dual_view_v2 first.")
with open(DV_PATH, encoding="utf-8") as f:
    dv = json.load(f)

print("dual_view_v2 best params (source of truth):")
for k, v in dv.items():
    if not k.startswith("_"):
        print(f"  {k:14s} = {v}")
print(f"  (best HPO-slice value: {dv['_best_value']:.4f}, {dv['_n_trials']} trials)")

In [ ]:
VARIANT = "baseline_v2"

# Baseline CV reads: batch_size, lr, weight_decay, warmup_ratio, epochs.
# lr is the (fixed) encoder learning rate; there is no separate head lr.
best = {
    "batch_size":   dv["batch_size"],
    "lr":           dv["encoder_lr"],
    "weight_decay": dv["weight_decay"],
    "warmup_ratio": dv["warmup_ratio"],
    "epochs":       dv["epochs"],
    "_variant":       VARIANT,
    "_derived_from":  "dual_view_v2",
    "_note":          "encoder fixed + epochs transferred (identical training setup); no search",
    "_best_value":    dv["_best_value"],
    "_n_trials":      0,
}

out = best_params_path(VARIANT)
with open(out, "w", encoding="utf-8") as f:
    json.dump(best, f, indent=2, ensure_ascii=False)
print(json.dumps(best, indent=2, ensure_ascii=False))
print("\nsaved:", out)

assert_test_untouched(globals())